# 03 - Context-Aware Report Sections

**ChatGPT Track**  
**allen-lab-report-tool**

This notebook converts Allen Lab context + source metadata into structured report sections.

Notebook 03 pipeline:

```text
Allen context artifact
+ source metadata artifact
→ report sections
→ JSON + Markdown export
```

Expected inputs:

```text
results/chatgpt/allen_lab_context.json
results/chatgpt/source_metadata.json
```

Expected outputs:

```text
results/chatgpt/report_sections.json
reports/chatgpt/report_sections.md
```


In [ ]:
# ================================================
# SETUP: Colab + local
# ================================================
from pathlib import Path
import json
import sys
import subprocess
from datetime import datetime, timezone

REPO_NAME = "allen-lab-report-tool"
REPO_URL = "https://github.com/thinkthoughts/allen-lab-report-tool.git"

cwd = Path.cwd()

if (cwd / "src" / "chatgpt" / "lab_context.py").exists():
    repo_root = cwd
elif cwd.name == "chatgpt" and cwd.parent.name == "notebooks":
    repo_root = cwd.parents[1]
elif (cwd / REPO_NAME / "src" / "chatgpt" / "lab_context.py").exists():
    repo_root = cwd / REPO_NAME
else:
    print("Repo not found in current runtime. Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo_root = cwd / REPO_NAME

src_path = repo_root / "src"
results_dir = repo_root / "results" / "chatgpt"
reports_dir = repo_root / "reports" / "chatgpt"

results_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("cwd:", cwd)
print("repo_root:", repo_root)
print("results_dir:", results_dir)
print("reports_dir:", reports_dir)


## 1. Load Input Artifacts

Notebook 03 expects Notebook 01 and Notebook 02 outputs. If they are missing, run those notebooks first.


In [ ]:
context_path = results_dir / "allen_lab_context.json"
metadata_path = results_dir / "source_metadata.json"

if not context_path.exists():
    raise FileNotFoundError(f"Missing {context_path}. Run Notebook 01 first.")

if not metadata_path.exists():
    raise FileNotFoundError(f"Missing {metadata_path}. Run Notebook 02 first.")

allen_context = json.loads(context_path.read_text(encoding="utf-8"))
source_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

print("Loaded:", context_path)
print("Loaded:", metadata_path)


## 2. Inspect Source + Context

In [ ]:
source_record = source_metadata["source_record"]
context_matches = source_metadata.get("context_matches", {})

print("Source title:", source_record["title"])
print("Institution:", source_record["institution"])
print("Matched focus areas:", context_matches.get("matched_focus_areas", []))
print("Matched equipment/platforms:", context_matches.get("matched_equipment_or_platforms", []))
print("Matched report priorities:", context_matches.get("matched_report_priorities", []))


## 3. Define Section Helpers

In [ ]:
def bullets(items):
    if not items:
        return "- (none listed)"
    return "\n".join(f"- {item}" for item in items)


def section(section_id, title, content, source_notes=None, tags=None):
    return {
        "section_id": section_id,
        "title": title,
        "content": content.strip(),
        "source_notes": source_notes or [],
        "tags": tags or [],
    }


def section_to_markdown(section_record):
    lines = [
        f"## {section_record['title']}",
        "",
        section_record["content"],
        "",
    ]

    if section_record.get("source_notes"):
        lines.extend(["### Source Notes", ""])
        lines.append(bullets(section_record["source_notes"]))
        lines.append("")

    if section_record.get("tags"):
        lines.extend(["### Tags", ""])
        lines.append(bullets(section_record["tags"]))
        lines.append("")

    return "\n".join(lines)


## 4. Generate Report Sections

These sections are still lightweight. Notebook 04 will combine them into a fuller report.


In [ ]:
matched_focus = context_matches.get("matched_focus_areas", [])
matched_platforms = context_matches.get("matched_equipment_or_platforms", [])
matched_priorities = context_matches.get("matched_report_priorities", [])

sections = []

sections.append(section(
    section_id="source_overview",
    title="Source Overview",
    content=f"""
**Title:** {source_record['title']}

**Institution:** {source_record['institution']}

**Source type:** {source_record['source_type']}

**Source URL:** {source_record.get('source_url') or '(not set yet)'}

{source_record['abstract_or_summary']}
""",
    source_notes=[
        "Built from source_metadata.json generated by Notebook 02.",
        "Source record can be replaced with a specific Allen Lab paper, dataset page, or methods note.",
    ],
    tags=["source", "metadata", "overview"],
))

sections.append(section(
    section_id="allen_context_match",
    title="Allen Lab Context Match",
    content=f"""
This source record connects to the Allen Lab context profile through direct term matches.

### Matched focus areas

{bullets(matched_focus)}

### Matched equipment or platforms

{bullets(matched_platforms)}

### Matched report priorities

{bullets(matched_priorities)}
""",
    source_notes=[
        "Matches use simple text overlap between source metadata and Allen context terms.",
        "Later notebooks can replace direct matching with richer parsing or review workflows.",
    ],
    tags=["context", "matching", "allen-lab"],
))

sections.append(section(
    section_id="methods_platform_notes",
    title="Methods and Platform Notes",
    content=f"""
The current source metadata mentions or aligns with these possible platform categories:

{bullets(matched_platforms)}

These platform terms help shape report sections around methods, measurements, datasets, and reproducibility.
""",
    source_notes=[
        "Platform notes are generated from context matches, not full methods extraction.",
        "Notebook 04 can expand this into a report-ready Methods and Reproducibility section.",
    ],
    tags=["methods", "platforms", "reproducibility"],
))

sections.append(section(
    section_id="report_priorities",
    title="Report Priorities",
    content=f"""
The current source record aligns with these report priorities:

{bullets(matched_priorities)}

For labreports.app, these priorities define what a useful continuation report should preserve: source structure, traceable methods, reusable summaries, and public-facing clarity.
""",
    source_notes=[
        "Report priorities come from the Allen Lab context profile plus source metadata matches.",
    ],
    tags=["reporting", "priorities", "handoff"],
))

sections.append(section(
    section_id="follow_up_questions",
    title="Follow-up Report Questions",
    content="""
Useful next questions for the report pipeline:

- What specific Allen Lab source should replace the placeholder record?
- Which methods section, dataset page, or source URL should anchor the first public demo?
- Which figures or tables should be reproduced or summarized?
- Which report outputs should be prepared for GitHub, white paper, or labreports.app demo use?
- Which sections need human review before outreach?
""",
    source_notes=[
        "These questions prepare Notebook 04 and later handoff demos.",
    ],
    tags=["next-steps", "review", "demo"],
))

sections


## 5. Display Report Section Index

In [ ]:
import pandas as pd

sections_df = pd.DataFrame([
    {
        "section_id": s["section_id"],
        "title": s["title"],
        "tags": ", ".join(s["tags"]),
    }
    for s in sections
])

display(sections_df)


## 6. Build Report Sections Artifact

In [ ]:
report_sections_artifact = {
    "generator_track": "chatgpt",
    "source_file": "notebooks/chatgpt/03_context_aware_report_sections.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "inputs": {
        "allen_context": str(context_path.relative_to(repo_root)),
        "source_metadata": str(metadata_path.relative_to(repo_root)),
    },
    "source_title": source_record["title"],
    "sections": sections,
}

report_sections_artifact


## 7. Export JSON

In [ ]:
sections_json_path = results_dir / "report_sections.json"

with sections_json_path.open("w", encoding="utf-8") as f:
    json.dump(report_sections_artifact, f, indent=2, ensure_ascii=False)

print("Wrote:", sections_json_path)


## 8. Export Markdown

In [ ]:
section_markdown = "\n---\n\n".join(section_to_markdown(s) for s in sections)

report_sections_md = f"""# Context-Aware Report Sections

**Generator track:** ChatGPT  
**Notebook:** `notebooks/chatgpt/03_context_aware_report_sections.ipynb`  
**Source title:** {source_record['title']}

## Inputs

- `{str(context_path.relative_to(repo_root))}`
- `{str(metadata_path.relative_to(repo_root))}`

---

{section_markdown}
"""

sections_md_path = reports_dir / "report_sections.md"
sections_md_path.write_text(report_sections_md, encoding="utf-8")

print("Wrote:", sections_md_path)


## 9. Confirm Exports

In [ ]:
print(sections_json_path.read_text(encoding="utf-8"))


In [ ]:
print(sections_md_path.read_text(encoding="utf-8"))


## 10. Summary

Notebook 03 turns the Allen context + source metadata artifacts into structured report sections.

Current outputs:

```text
results/chatgpt/report_sections.json
reports/chatgpt/report_sections.md
```

**Next:** Notebook 04 — Full Lab Report Export.
